In [1]:
%pip install -q onnx onnxscript onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 7.8 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install --upgrade onnxruntime onnx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 35.8 MB/s eta 0:00:00
  Attempting uninstall: onnx
    Found existing installation: onnx 1.21.0
    Uninstalling onnx-1.21.0:
      Successfully uninstalled onnx-1.21.0
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import glob
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
import onnx
import onnxruntime as ort

# ==========================================
# 1. TÌM VÀ NẠP DỮ LIỆU
# ==========================================
npz_search = glob.glob('/kaggle/input/**/*.npz', recursive=True)
data_path = npz_search[0]
raw_data = np.load(data_path)

X_train = torch.FloatTensor(raw_data['X_train']).flatten(start_dim=1)
y_train = torch.FloatTensor(raw_data['y_train'])
X_val   = torch.FloatTensor(raw_data['X_val']).flatten(start_dim=1)
y_val   = torch.FloatTensor(raw_data['y_val'])

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)

# ==========================================
# 2. KIẾN TRÚC MẠNG
# ==========================================
class StableLayerNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.bias   = nn.Parameter(torch.zeros(dim))
        self.eps    = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        mean = x.mean(dim=-1, keepdim=True)
        var  = ((x - mean) ** 2).mean(dim=-1, keepdim=True)
        return self.weight * (x - mean) / torch.sqrt(var + self.eps) + self.bias

class BlendshapeCorrector_MLP(nn.Module):
    def __init__(self, input_dim=287, hidden_dim=256, output_dim=52):
        super().__init__()
        self.input_norm = StableLayerNorm(input_dim)
        
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim // 2)
        self.fc3 = nn.Linear(hidden_dim // 2, output_dim)
        
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        
    def forward(self, x):
        x = self.input_norm(x)
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.dropout(self.relu(self.fc2(x)))
        return torch.sigmoid(self.fc3(x))

# ==========================================
# 3. HUẤN LUYỆN MODEL
# ==========================================
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
output_dim = y_train.shape[1]
model = BlendshapeCorrector_MLP(input_dim=287, output_dim=output_dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

best_val_loss = float('inf')
epochs = 50

for epoch in range(epochs):
    model.train()
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(batch_X.to(device)), batch_y.to(device))
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_loss = criterion(model(X_val.to(device)), y_val.to(device)).item()

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), '/kaggle/working/best_model.pth')

print(f"[SUCCESS] Huấn luyện xong! Output shape đang là: (Batch_size, {output_dim})")

# ==========================================
# 4. XUẤT ONNX SẠCH (HỖ TRỢ PYTORCH 2.4+ MỚI NHẤT)
# ==========================================
model.load_state_dict(torch.load('/kaggle/working/best_model.pth', weights_only=True))
model.eval()

dummy_input = torch.randn(1, 287).to(device)
onnx_path = '/kaggle/working/blendshape_corrector.onnx'

print("[ĐÓNG GÓI] Đang xuất file ONNX...")

# Sử dụng chính xác chuẩn Opset 18 của PyTorch mới
torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    export_params=True,
    opset_version=18, 
    do_constant_folding=True,
    input_names=['input_tensor'],
    output_names=['corrected_blendshapes']
)

print(f"[OK] Đã lưu file vật lý tại: {onnx_path}")

# Test thử file ONNX vừa xuất
sess = ort.InferenceSession(onnx_path)
dummy_np = np.random.randn(1, 287).astype(np.float32)
result = sess.run(['corrected_blendshapes'], {'input_tensor': dummy_np})
print(f"[VERIFIED] ONNX chạy thành công 100%! Output shape: {result[0].shape}")

[SUCCESS] Huấn luyện xong! Output shape đang là: (Batch_size, 1)
[ĐÓNG GÓI] Đang xuất file ONNX...
[torch.onnx] Obtain model graph for `BlendshapeCorrector_MLP([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `BlendshapeCorrector_MLP([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[OK] Đã lưu file vật lý tại: /kaggle/working/blendshape_corrector.onnx
[VERIFIED] ONNX chạy thành công 100%! Output shape: (1, 1)


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
